**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/elliott-2026"], check=True)
    os.chdir("/content/encore/entries/elliott-2026")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Elliott, Andrews et al. (2026): spike-based claims

Paper: Elliott, Andrews et al., "Microscale maps of bursting dynamics across human hippocampal slices
from patients with epilepsy", J. Neurophysiol. (2026), doi:10.1152/jn.00217.2025. The claims file
(`claims/elliott-2026.yaml`) was written from the accepted author manuscript in PubMed Central
(PMC12931417) and its supplement (Extended Data).

Dataset: DANDI:001132, version 0.241130.1903 (doi:10.48324/dandi.001132/0.241130.1903), the raw
HD-MEA recordings of Andrews et al. 2024. The paper cites the dandiset without a version. Two
versions are published, 0.241119.1801 and 0.241130.1903; both list the same 69 assets with the same
total size, so the most recent published version is pinned, as the protocol says for a paper that
cites none.

Outside input: the curated spike sortings the paper analysed are not in the dandiset. They are read
into memory from the paper's repository, github.com/braingeneers/human_hippocampus at commit
0f83f704ee4c9b4d8d7f75eb53d94f525c85abb2 (the claims file's opening comment). Which sorting file is
which recording is taken from the paper's latency notebooks (`figures/figure_3/latencies/t_test_try30_try3/`),
which load one file per recording (see `hippo.RECORDINGS`).

This notebook holds the claims that need spikes only. The claims that need the raw voltage (LFP,
theta, phase locking) are in `lfp_elliott_2026.ipynb`.

| Claim | Here | Status |
|---|---|---|
| sorting-dandiset-link (link) | yes | tested |
| table1-unit-counts | yes | tested |
| fig1g-ka-synchronized-bursting | yes | tested |
| methods-burst-counts | yes | tested |
| methods-rate-increase-after-ka | yes | tested |
| methods-onset-latency | yes | blocked (metadata_insufficient: no time of KA application) |
| s4-ka-no-epileptiform | yes | tested |
| fig2c-sttc-two-clusters | yes | tested |
| ext9-sttc-eigenvector-gradient | yes | tested |
| ext9-sttc-beats-correlation | yes | tested |
| fig3a-significant-pairs | yes | tested |
| fig3a-bipolar-angle-histograms | yes | tested |
| ext10-baseline-same-circuit | yes | tested |
| table3-geometry-test | yes | tested |
| ext3, fig1j, ext4b, ext4, fig2a, table2 (two claims), fig2b | lfp notebook | |
| methods-rolling-wave-counts | lfp notebook | blocked (methods_underspecified) |
| ext12-simulation-recurrence | yes | blocked (not_runnable: the simulation's conda environment, section at the end) |

In [ ]:
import time, zlib, resource
import multiprocessing as mp
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from concurrent.futures import ProcessPoolExecutor
from scipy import stats
from scipy.cluster import hierarchy
from scipy.spatial.distance import pdist
import hippo as H
T0 = time.time()
print('dandiset', H.DANDISET_ID, 'version', H.DANDISET_VERSION, '; code commit', H.CODE_COMMIT)

## Data check

The dandiset's files hold one ElectricalSeries each (raw voltage, uint16 with a conversion to volts,
20 kHz) and an electrode table with positions (`rel_x`, `rel_y`). They hold no units, no epochs, no
annotations and no stimulus or drug timing. Subjects are named by slice (`sub-<n><patient>-<k>`), and
the session start times are all the conversion date (2024-09-28).

In [ ]:
assets = H.list_assets()
A = {a['path'].split('/')[0]: a for a in assets}
def meta(a):
    f = H.open_nwb(a['asset_id'])
    d = f['acquisition/ElectricalSeries/data']
    e = H.electrodes(f)
    g = lambda k: f[k][()].decode() if k in f else ''
    return dict(subject=a['path'].split('/')[0], n_samples=d.shape[0], n_channels=d.shape[1],
                rate=float(f['acquisition/ElectricalSeries/starting_time'].attrs['rate']),
                conversion=float(d.attrs['conversion']), unit=d.attrs['unit'],
                minutes=d.shape[0] / H.FS / 60, age=g('general/subject/age'), sex=g('general/subject/sex'),
                start=g('session_start_time')[:10], groups=list(f.keys()), units='units' in f,
                processing=list(f['processing'].keys()), intervals='intervals' in f,
                stim=list(f['stimulus/presentation'].keys()),
                locations=sorted({v.decode() if isinstance(v, bytes) else str(v) for v in f['general/extracellular_ephys/electrodes/location'][:]}), x_range=(e.x.min(), e.x.max()), y_range=(e.y.min(), e.y.max())), e
with ProcessPoolExecutor(4, mp_context=mp.get_context('fork')) as ex:
    res = list(ex.map(meta, assets))
M = pd.DataFrame([r[0] for r in res]).sort_values('subject').reset_index(drop=True)
ELEC = {r[0]['subject']: r[1] for r in res}
print(len(M), 'files;', M.subject.str.extract(r'sub-(\d+[A-Z])')[0].nunique(), 'slices')
print('rates', M.rate.unique(), '; units tables', M.units.sum(), '; processing modules', M.processing.map(len).sum(),
      '; intervals', M.intervals.sum(), '; stimulus series', M.stim.map(len).sum(), '; session dates', M.start.unique())
print('electrode locations in the electrodes tables of all files:', sorted(set(sum(M.locations, []))))
print('conversion (V per count)', M.conversion.unique(), M.unit.unique())
print('channels', M.n_channels.min(), 'to', M.n_channels.max(), '; electrode x', M.x_range.map(lambda r: r[0]).min(), 'to',
      M.x_range.map(lambda r: r[1]).max(), 'um; y', M.y_range.map(lambda r: r[0]).min(), 'to', M.y_range.map(lambda r: r[1]).max(), 'um')
pd.set_option('display.width', 200)
print(M[['subject', 'n_channels', 'minutes', 'age', 'sex']].to_string())

Recordings deposited more than once: every pair of files of the same slice whose durations differ
by less than 1 s is compared on one second of raw data (10 to 11 s, first 64 channels): the
fraction of identical samples, and the mean correlation of the sample-to-sample changes (which
removes slow shared drift). Independent recordings give values near 0.

In [ ]:
from itertools import combinations
M['slice'] = M.subject.str.extract(r'sub-(\d+[A-Z])')[0]
pairs = [(a.subject, b.subject) for _, g in M.groupby('slice') for (_, a), (_, b) in combinations(g.iterrows(), 2)
         if abs(a.minutes - b.minutes) * 60 < 1.0 and a.n_channels == b.n_channels]
def snippet(sub):
    f = H.open_nwb(A[sub]['asset_id'])
    return sub, f['acquisition/ElectricalSeries/data'][200000:220000, :64].astype(np.float64)
SNIP = dict(ProcessPoolExecutor(4, mp_context=mp.get_context('fork')).map(snippet, sorted({s for p in pairs for s in p})))
rows = []
for a, b in pairs:
    x, y = SNIP[a], SNIP[b]
    dx, dy = np.diff(x, axis=0), np.diff(y, axis=0)
    cc = [np.corrcoef(dx[:, k], dy[:, k])[0, 1] for k in range(64) if dx[:, k].std() > 0 and dy[:, k].std() > 0]
    rows.append(dict(file_a=a, file_b=b, minutes=round(M.set_index('subject').minutes[a], 3), identical_samples=np.mean(x == y),
                     diff_corr=np.mean(cc), all_identical=bool(np.array_equal(x, y))))
DUP = pd.DataFrame(rows)
print(DUP.round(3).to_string(index=False))
print('pairs compared', len(DUP), '; pairs with more than half the samples identical', int((DUP.identical_samples > 0.5).sum()))

The six pairs of sub-3C-1 to sub-3C-4 share most samples in that second. To tell copies from a
shared artifact, the six sub-3C files are compared over their first 40 s and all 972 channels,
second by second: the fraction of identical samples, the fraction of samples at the limits of the
10-bit converter (0 or 1023 counts), the correlation of the 1 kHz means (20-sample means, the LFP)
and of the sample-to-sample changes, per channel, averaged over channels. sub-3C-0 (0.91 min) and
sub-3C-5 (1.17 min) have other lengths and serve as references. Copies would share the LFP at
every second; seconds where any file is saturated are reported apart.

In [ ]:
FILES3C = [f'sub-3C-{i}' for i in range(6)]
P3C = list(combinations(range(6), 2))
N40, STEP8 = int(40 * H.FS), int(8 * H.FS)
def dup_block(blk):
    ds = [H.open_nwb(A[s]['asset_id'])['acquisition/ElectricalSeries/data'] for s in FILES3C]
    c0, c1 = blk * 64, min(blk * 64 + 64, ds[0].shape[1])
    ident = np.zeros((len(P3C), 40)); rail = np.zeros((6, 40))
    lcorr = np.zeros((len(P3C), 40, c1 - c0)); dcorr = np.zeros((len(P3C), 40, c1 - c0))
    z = lambda v: (v - v.mean(0)) / (v.std(0) + 1e-9)
    for a in range(0, N40, STEP8):
        X = [dd[a:a + STEP8, c0:c1].astype(np.int32) for dd in ds]
        for sec in range(8):
            seg = [x[sec * 20000:(sec + 1) * 20000] for x in X]
            L = [z(x.reshape(-1, 20, x.shape[1]).mean(1)) for x in seg]   # 1 kHz means (LFP)
            D = [z(np.diff(x, axis=0).astype(float)) for x in seg]        # sample-to-sample changes
            s = a // 20000 + sec
            for i, x in enumerate(seg):
                rail[i, s] += np.sum((x == 0) | (x == 1023))
            for k, (i, j) in enumerate(P3C):
                ident[k, s] += np.sum(seg[i] == seg[j])
                lcorr[k, s] = np.mean(L[i] * L[j], 0); dcorr[k, s] = np.mean(D[i] * D[j], 0)
    return ident, rail, lcorr, dcorr
t = time.time()
with ProcessPoolExecutor(4, mp_context=mp.get_context('fork')) as ex:
    R = list(ex.map(dup_block, range(int(np.ceil(972 / 64)))))
nch = 972
ident = sum(r[0] for r in R) / (20000 * nch); rail = sum(r[1] for r in R) / (20000 * nch)
lcorr = np.concatenate([r[2] for r in R], axis=2).mean(2); dcorr = np.concatenate([r[3] for r in R], axis=2).mean(2)
sat = (rail > 0.1).any(0)
print(f'{time.time()-t:.0f} s; first 40 s, all {nch} channels, per second')
print('fraction of samples at the ADC limits (0 or 1023 counts), seconds where it exceeds 0.1:')
for i, s in enumerate(FILES3C):
    print(f'  {s} ({M.set_index("subject").minutes[s]:.2f} min): seconds {np.where(rail[i] > 0.1)[0].tolist()}; largest {rail[i].max():.2f}; elsewhere {rail[i][~sat].max():.4f} at most')
rows = []
for k, (i, j) in enumerate(P3C):
    rows.append(dict(file_a=FILES3C[i], file_b=FILES3C[j], same_length=bool(abs(M.set_index('subject').minutes[FILES3C[i]] - M.set_index('subject').minutes[FILES3C[j]]) * 60 < 1),
                     identical_all=ident[k].mean(), identical_saturated_s=ident[k][sat].mean(), identical_other_s=ident[k][~sat].mean(),
                     identical_other_s_max=ident[k][~sat].max(), lfp_corr_other_s=np.median(lcorr[k][~sat]), lfp_corr_other_s_max=lcorr[k][~sat].max(),
                     diff_corr_other_s=np.median(dcorr[k][~sat]), lfp_corr_above_half_s=np.where(lcorr[k] > 0.5)[0].tolist()))
DUP3C = pd.DataFrame(rows)
print('saturated seconds (any file above 0.1):', np.where(sat)[0].tolist())
print(DUP3C.round(3).to_string(index=False))
print(f'peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1e6:.1f} GB')

In [ ]:
t = time.time()
SORT = {r: H.load_sorting(r) for r in H.RECORDINGS}
rows = []
for r, s in SORT.items():
    rates = np.array([len(x) for x in s['trains']]) / (s['length_ms'] / 1000)
    rows.append(dict(rec=r, slice=H.RECORDINGS[r]['slice'], cond=H.RECORDINGS[r]['cond'], units=s['n'],
                     spikes=sum(len(x) for x in s['trains']), span_min=s['length_ms'] / 60000,
                     first_spike_s=min(x[0] for x in s['trains'] if len(x)) / 1000,
                     median_rate_hz=np.median(rates), x_um=f"{s['pos'][:,0].min():.0f}-{s['pos'][:,0].max():.0f}",
                     y_um=f"{s['pos'][:,1].min():.0f}-{s['pos'][:,1].max():.0f}",
                     unsorted_trains=sum(np.any(np.diff(x) < 0) for x in s['trains']),
                     empty=sum(len(x) == 0 for x in s['trains'])))
SUM = pd.DataFrame(rows)
print(SUM.to_string(index=False))
print(f'loaded in {time.time()-t:.0f} s')

Which sorting file is which recording, printed from the paper's latency notebooks
(`figures/figure_3/latencies/t_test_try30_try3/`) at the pinned commit: the line of each notebook
that loads a sorting, and whether it names the file used here (`hippo.RECORDINGS`).

In [ ]:
LAT = 'figures/figure_3/latencies/t_test_try30_try3/'
LAT_NB = {1: 'hc112823_acid.ipynb', 2: 'hc112823_base.ipynb', 3: 'elliott_chip21769_acid.ipynb', 4: 'elliott_chip21769_base.ipynb',
          5: 'hc111_base.ipynb', 6: 'hc110723_acid.ipynb', 7: 'hc328.ipynb', 8: 'hc52_base.ipynb'}
for r, name in LAT_NB.items():
    hits = [h for h in H.code_lines(LAT + name, r'data/ephys') if not h[2].startswith('#')]
    print(r, H.RECORDINGS[r]['slice'], H.RECORDINGS[r]['cond'], f'| {name} cell {hits[0][0]} line {hits[0][1]}: {hits[0][2][:170]}',
          '| names the file used here:', any(H.RECORDINGS[r]['path'] in h[2] for h in hits))
others = sorted(p.split('/')[-1] for p in H.code_tree() if p.startswith(LAT) and p.endswith('.ipynb') and p.split('/')[-1] not in LAT_NB.values())
for name in others:
    print('not used:', name, [h[2][:170] for h in H.code_lines(LAT + name, r'data/ephys') if not h[2].startswith('#')])

## sorting-dandiset-link (link claim added by the author)

> The curated sortings of the paper's repository can be linked to the dandiset's raw recordings: each
> recording's units sit on the electrodes of one dandiset file, and its spike times align with the
> raw voltage of that file.

Matching: (1) the fraction of unit positions that fall on an electrode position of each file;
(2) in the files whose layout matches, an offset scan (`hippo.offset_scan`): the spike trains of
the three most active units are correlated with the high-passed voltage of their nearest electrode
at every offset over the whole file. A sorting from that file gives a peak far below the chance
level (the 100th most negative value), at the offset where the sorting starts.

In [ ]:
layout = {}
for r, s in SORT.items():
    P = set(map(tuple, np.round(s['pos'], 1)))
    frac = {sub: len(P & set(zip(np.round(e.x, 1), np.round(e.y, 1)))) / len(P) for sub, e in ELEC.items()}
    full = sorted(k for k, v in frac.items() if v == 1.0)
    layout[r] = full
    best_other = max(v for k, v in frac.items() if v < 1.0)
    print(r, H.RECORDINGS[r]['slice'], H.RECORDINGS[r]['cond'], '-> all units on electrodes of', full,
          f'; best other file {best_other:.2f}')

In [ ]:
t = time.time()
jobs = [(r, sub) for r in SORT for sub in layout[r]]
def scan(job):
    r, sub = job
    return (r, sub) + H.offset_scan(A[sub]['asset_id'], SORT[r])
with ProcessPoolExecutor(4, mp_context=mp.get_context('fork')) as ex:
    SCAN = pd.DataFrame(list(ex.map(scan, jobs)), columns=['rec', 'file', 'offset_s', 'peak', 'chance'])
SCAN['ratio'] = SCAN.peak / SCAN.chance
print(SCAN.round(3).to_string(index=False))
print(f'scan took {time.time()-t:.0f} s')

In [ ]:
LINK = {}
for r in SORT:
    d = SCAN[SCAN.rec == r]
    hit = d[d.ratio > 10]
    if len(hit) == 1:
        LINK[r] = dict(file=hit.file.iloc[0], offset_s=round(float(hit.offset_s.iloc[0]), 3))
    print(r, H.RECORDINGS[r]['slice'], H.RECORDINGS[r]['cond'], '->', LINK.get(r, 'no file matched'),
          f'(best ratio {d.ratio.max():.1f}, files tried {len(d)})')
print('linked', len(LINK), 'of', len(SORT))

The scan correlates at every offset, including offsets where the sorting overlaps the file only in
part (full-mode correlation), so a file that holds any stretch of a sorting's recording gives a peak.
Scanning S5 against each sub-3C file in turn therefore covers the case where the S5 recording is
split over several sub-3C files joined in index order. To show that the scan finds a short piece,
the S3 sorting is scanned against only 100 to 140 s of its file: 40 s, the length of the shortest
sub-3C files.

In [ ]:
t = time.time()
part = H.offset_scan(A[LINK[5]['file']]['asset_id'], SORT[5], t0=100, t1=140)
print(f"S3 sorting against 100 to 140 s of {LINK[5]['file']}: offset {part[0]:.3f} s from 100 s (-100 expected), "
      f"peak {part[1]:.0f}, chance {part[2]:.1f}, ratio {part[1] / part[2]:.1f}; {time.time() - t:.0f} s")
d7 = SCAN[SCAN.rec == 7]
print('S5 against the sub-3C files:', d7[['file', 'ratio']].round(2).to_dict('records'))
print('sub-3C durations (min):', M.set_index('subject').minutes[[f for f in d7.file]].round(2).to_dict(), '; S5 sorting span',
      round(SORT[7]['length_ms'] / 60000, 2), 'min')

**Result: partially reproduced.** Every unit position of each sorting is an electrode position of one slice's files (fraction 1.00; the best other file 0.04 to 0.43), and each sorting file is the one the paper's latency notebook for that recording loads. The offset scan finds recordings 1 to 6 in single files with peaks 74 to 169 times the chance level: S1 KA in sub-10F-6 and S1 baseline in sub-10F-0 (from 20.0 s), S2 KA in sub-11G-2 and S2 baseline in sub-11G-4, S3 in sub-1A-0, and S4 in sub-9E-2 (from 30.0 s). Recordings 7 (S5) and 8 (S6) are not found in any file with their electrode layout (sub-3C and sub-8D files, best ratio 1.3 and 1.4). The scan finds the S3 sorting in a 40 s piece of its file (ratio 39 at the expected offset), and every sub-3C file is at least 40 s long, so no sub-3C file holds a stretch of the S5 recording, joined to others or not. 6 of 8 recordings are linked.

## table1-unit-counts

> The eight recordings analysed hold 212, 249, 180, 51, 87, 109, 136 and 100 sorted units
> (recordings 1 to 8) and last 6, 5, 8.5, 5, 7, 2.2, 6.4 and 1.8 minutes.

Units are counted in the curated sortings; durations are read from the dandiset file each sorting
is linked to (criterion). The span covered by the sorting is shown beside it.

In [ ]:
exp_units = [212, 249, 180, 51, 87, 109, 136, 100]
exp_min = [6, 5, 8.5, 5, 7, 2.2, 6.4, 1.8]
rows = []
for r in SORT:
    fmin = M.set_index('subject').minutes.get(LINK[r]['file']) if r in LINK else np.nan
    u, eu, em = SORT[r]['n'], exp_units[r-1], exp_min[r-1]
    rows.append(dict(rec=r, units=u, expected_units=eu, units_ok=abs(u-eu) <= 0.1*eu, file=LINK.get(r, {}).get('file', 'none'),
                     file_min=fmin, sorting_span_min=SORT[r]['length_ms']/60000, expected_min=em,
                     file_min_ok=bool(abs(fmin-em) <= 0.1*em) if fmin == fmin else False,
                     span_ok=abs(SORT[r]['length_ms']/60000-em) <= 0.1*em))
T1 = pd.DataFrame(rows)
T1['both_ok'] = T1.units_ok & T1.file_min_ok
print(T1.round(2).to_string(index=False))
print('recordings matching on units and dandiset duration:', int(T1.both_ok.sum()), 'of 8; units alone', int(T1.units_ok.sum()),
      '; units and sorting span', int((T1.units_ok & T1.span_ok).sum()))

**Result: not reproduced.** All eight unit counts equal the paper's (212, 249, 180, 51, 87, 109, 136, 100), and the spans of the sortings match all eight durations. On the dandiset's files, five recordings match on both units and duration; S4's file lasts 2.67 min against 2.2 (the sorting covers its 30 to 160 s), and the files of S5 and S6 are not in the dandiset, so 5 of 8 match where the criterion asks for at least 6.

## fig1g-ka-synchronized-bursting

> After kainic acid, slice S1 shows synchronized rhythmic bursting, with about 6 to 8 bursts
> grouped within larger burst epochs (superbursts).

Population rate as the Methods define it (sigma 10 and 20 ms). Burst thresholds, the readings the
criterion names: (a) twice the mean over the pre-KA part, (b) mean plus 3 SD over the whole
recording. The dandiset holds no time of KA application (data check) and bursting starts within the
first seconds of both KA recordings, so the pre-KA part of reading (a) is taken as the separate
baseline recording of the same slice (recordings 2 and 4, recorded before KA; Results, paragraph 10,
"the non-epileptiform-like recordings prior to administering kainic acid"); for slices without a
separate baseline (S3 to S6) the recording's own mean is used. Peaks closer than 50 ms are merged,
epochs are runs of bursts separated by gaps under 5 s. Counts: (i) median bursts per epoch over the
epochs that start in the first 150 s, (ii) bursts in the first epoch.

In [ ]:
def burst_table(r, sigma, reading):
    s = SORT[r]
    rate = H.population_rate(s['trains'], s['length_ms'], sigma)
    if reading == 'a':
        ref = {1: 2, 3: 4}.get(r)
        base = H.population_rate(SORT[ref]['trains'], SORT[ref]['length_ms'], sigma).mean() if ref else rate.mean()
        thr = 2 * base
    else:
        thr = rate.mean() + 3 * rate.std()
    bt, ep = H.detect_bursts(rate, thr)
    return rate, thr, bt, ep

READINGS = [(sg, rd) for rd in ('a', 'b') for sg in (10, 20)]
BURSTS = {(r, sg, rd): burst_table(r, sg, rd) for r in SORT for sg, rd in READINGS}
rows = []
for sg, rd in READINGS:
    rate, thr, bt, ep = BURSTS[(1, sg, rd)]
    early = [len(e) for e in ep if e[0] < 150]
    syn = H.burst_synchrony(SORT[1]['trains'], bt, SORT[1]['length_ms'])
    rows.append(dict(reading=rd, sigma=sg, threshold_hz=thr, mean_rate=rate.mean(), bursts=len(bt), epochs=len(ep),
                     epoch_sizes_first150=early, median_first150=np.median(early) if early else np.nan,
                     first_epoch=len(ep[0]) if ep else 0, first_epoch_start_s=ep[0][0] if ep else np.nan,
                     median_frac_units_up=np.median(syn) if len(syn) else np.nan,
                     frac_bursts_majority=np.mean(syn > 0.5) if len(syn) else np.nan))
F1 = pd.DataFrame(rows)
F1['in_range_median'] = F1.median_first150.between(5.4, 8.8)
F1['in_range_first'] = F1.first_epoch.between(5.4, 8.8)
print(F1.round(2).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 5))
for a, sg, rd in [(ax[0], 10, 'b'), (ax[1], 20, 'a')]:
    rate, thr, bt, ep = BURSTS[(1, sg, rd)]
    tt = np.arange(len(rate)) / 1000
    m = tt < 150
    a.plot(tt[m], rate[m], lw=0.5); a.axhline(thr, color='r', lw=0.8)
    a.plot(bt[bt < 150], rate[(bt[bt < 150] * 1000).astype(int)], 'k.', ms=3)
    a.set_ylabel('Hz per unit'); a.set_title(f'S1 KA, first 150 s, sigma {sg} ms, reading ({rd})', fontsize=9)
    print(f'reading ({rd}) sigma {sg}: rate range {rate[m].min():.2f} to {rate[m].max():.1f} Hz in the first 150 s')
ax[1].set_xlabel('s'); plt.tight_layout(); plt.show()

In [ ]:
# the paper's code sets the S1 initiation sub-bursts by hand at 14.4 to 14.91 s (claims file, opening comment);
# where do the detected bursts and epochs fall relative to it?
for sg, rd in READINGS:
    rate, thr, bt, ep = BURSTS[(1, sg, rd)]
    print(f'({rd}) sigma {sg}: bursts before 14.4 s: {np.round(bt[bt < 14.4], 2)}; first burst at or after 14.4 s: {bt[bt >= 14.4][:1].round(2)}; '
          f'epoch starts (s): {np.round([e[0] for e in ep], 1)[:14]}')

In [ ]:
# variant outside the criterion, shown for the question to a reader: the code's S1 sub-bursts are 170 to 180 ms
# apart (14.4, 14.57, 14.75, 14.91 s), so a 50 ms merge can count sub-bursts as bursts; peaks closer than 250 or
# 500 ms merged instead. This does not decide the outcome.
for sg, rd in READINGS:
    rate, thr = BURSTS[(1, sg, rd)][:2]
    for merge in (250, 500):
        bt, ep = H.detect_bursts(rate, thr, merge_ms=merge)
        early = [len(e) for e in ep if e[0] < 150]
        print(f'({rd}) sigma {sg}, merge {merge} ms: {len(bt)} bursts, {len(ep)} epochs, sizes in the first 150 s {early}, '
              f'median {np.median(early):.1f}, first epoch {len(ep[0])}')

**Result: not reproduced.** Under none of the four readings does the count fall in 5.4 to 8.8: the median bursts per epoch in the first 150 s is 15 to 40, and the first epoch holds 2 to 4 bursts (it starts at 0.73 to 0.90 s, before the paper's initiation at 14.4 s). The median fraction of units whose rate rises at a burst is 0.49 to 0.53. Bursts are grouped: all readings give 12 epochs. Outside the criterion, merging peaks closer than 500 ms (the code's sub-bursts are 170 to 180 ms apart) gives a median of 7.5 to 10 bursts per epoch, and 7.5 and 8.0 under reading (b); this reading of a burst is asked of a reader and does not decide the outcome.

## methods-burst-counts

> Slice S1 had 67 bursts grouped into 11 superbursts over about 350 s; S2 had 47 bursts, highly
> synchronized for the first 50 s and sporadic for the next 300 s; in both slices bursts occurred for
> about 350 s.

Same readings as above. Duration of bursting is the time from the first to the last burst. For S2,
the duration of synchronized bursting is the span of the first epoch (a run of bursts with gaps
under 5 s), the reading of "highly synchronized for the first 50 seconds" that the burst
definitions allow.

In [ ]:
rows = []
for sg, rd in READINGS:
    _, _, b1, e1 = BURSTS[(1, sg, rd)]
    _, _, b3, e3 = BURSTS[(3, sg, rd)]
    vals = [len(b1), len(e1), b1[-1] - b1[0] if len(b1) else 0, len(b3), (e3[0][-1] - e3[0][0]) if e3 else 0, b3[-1] - b3[0] if len(b3) else 0]
    exp = [67, 11, 350, 47, 50, 350]
    ok = [abs(v - x) <= 0.1 * x for v, x in zip(vals, exp)]
    rows.append(dict(reading=rd, sigma=sg, S1_bursts=vals[0], S1_superbursts=vals[1], S1_span_s=vals[2], S2_bursts=vals[3],
                     S2_first_epoch_s=vals[4], S2_span_s=vals[5], S1_ok=sum(ok[:3]), S2_ok=sum(ok[3:]), S2_epochs=len(e3),
                     S2_epoch_sizes=[len(e) for e in e3]))
F2 = pd.DataFrame(rows)
print(F2.round(1).to_string(index=False))
print('values within tolerance per reading (of 3 each for S1 and S2):', list(zip(F2.reading, F2.sigma, F2.S1_ok, F2.S2_ok)))

In [ ]:
# the paper's code lists S1 seizure windows (claims file, opening comment); count the windows and the
# bursts each reading puts inside them
s1_win = [(9,28),(38,52),(69,83),(100,113),(138,151),(176,187),(211,220),(246,254)]
for sg, rd in READINGS:
    _, _, b1, e1 = BURSTS[(1, sg, rd)]
    inside = sum(((b1 >= a) & (b1 < b)).sum() for a, b in s1_win)
    print(f'({rd}) sigma {sg}: {inside} of {len(b1)} S1 bursts fall in the code\'s 8 seizure windows; epochs starting before 254 s: {sum(e[0] < 254 for e in e1)}')

**Result: not reproduced.** Under every reading S1 has 12 epochs (11 expected, within tolerance) over 344 s (within tolerance), but 124 to 353 bursts against 67. S2 has 89 to 157 bursts against 47, a first epoch of 19.4 s against 50, and bursts over 486 to 491 s against 350. No reading matches all S1 or all S2 values. In the first 254 s, where the code lists 8 seizure windows, every reading has 9 epochs, and 90 to 92% of S1 bursts fall in the code's windows.

## methods-rate-increase-after-ka

> Neural activity increased after kainic acid; the population firing rate rose by 0.5 spikes per
> second per unit in S1 and 0.9 in S2.

Mean rate per unit (spike count over duration, averaged over units). Readings: (a) the KA recording
after KA onset against before it: not available, as the dandiset and the sortings hold no time of KA
application; (b) the whole KA recording against the baseline recording of the same slice; (c) the
bursting period (first to last burst, each burst reading) against the baseline recording.

In [ ]:
def mean_rate(s, t0=0, t1=None):
    t1 = s['length_ms'] if t1 is None else t1
    return np.mean([np.sum((x >= t0) & (x < t1)) for x in s['trains']]) / ((t1 - t0) / 1000)
rows = []
for ka, base, exp in [(1, 2, 0.5), (3, 4, 0.9)]:
    rb = mean_rate(SORT[base])
    rows.append(dict(slice=H.RECORDINGS[ka]['slice'], reading='b', ka_rate=mean_rate(SORT[ka]), base_rate=rb,
                     increase=mean_rate(SORT[ka]) - rb, expected=exp))
    for sg, rd in READINGS:
        bt = BURSTS[(ka, sg, rd)][2]
        rk = mean_rate(SORT[ka], bt[0] * 1000, bt[-1] * 1000)
        rows.append(dict(slice=H.RECORDINGS[ka]['slice'], reading=f'c ({rd}, sigma {sg})', ka_rate=rk, base_rate=rb,
                         increase=rk - rb, expected=exp))
R1 = pd.DataFrame(rows)
R1['within_tol'] = (R1.increase - R1.expected).abs() <= 0.1 * R1.expected + 1e-9
print(R1.round(3).to_string(index=False))
# unit-matched variant is not possible: the KA and baseline sortings were curated separately (different unit counts)
print('units KA/baseline:', SORT[1]['n'], SORT[2]['n'], '|', SORT[3]['n'], SORT[4]['n'])

**Result: not reproduced.** The mean rate per unit is lower in the KA recordings than in the baseline recordings: S1 4.20 against 4.32 spikes/s (-0.12), S2 2.28 against 2.38 (-0.10); over the bursting periods the change is -0.03 to -0.05. Reading (a) cannot be run, as no time of KA application is recorded. The KA and baseline sortings were curated separately (212 and 249 units in S1, 180 and 51 in S2), so the comparison is between different unit sets.

## methods-onset-latency (blocked)

> Epileptiform-like activity began in S1 and S2 shortly after kainic acid was applied.

The criterion needs the time of KA application from the dandiset's metadata or annotations. The data
check above shows that the files hold no epochs, intervals, annotations or stimulus series, and the
session descriptions are empty, so the claim is blocked as `metadata_insufficient`. For reference,
the first detected burst is printed (it bounds the latency only if KA was applied at or before the
recording start, which nothing states).

In [ ]:
for ka in (1, 3):
    print(H.RECORDINGS[ka]['slice'], {f'({rd}) sigma {sg}': round(float(BURSTS[(ka, sg, rd)][2][0]), 2) for sg, rd in READINGS})

## s4-ka-no-epileptiform

> Slice S4 received kainic acid but showed no epileptiform-like behaviour, and none of S3 to S6
> showed seizure-like activity.

A claim of absence. The criterion's test is synchronized bursting sustained for more than 10 s,
with bursts as in fig1g-ka-synchronized-bursting under each of its readings; the S1 and S2 KA
recordings must show it under the same reading. "Synchronized" is taken from the fig1g criterion,
which defines a synchronous burst as one that raises the rate of most units: a burst is kept when
more than half of the units fire above their mean rate within 50 ms of its peak
(`hippo.burst_synchrony`). Sustained is read as an epoch of kept bursts (gaps under 5 s) lasting
more than 10 s from its first to its last burst. The fraction is also run at 0.4 and 0.6, and with
every burst kept (the first version of this analysis, which dropped "synchronized"). The LFP part
of the paper's classification is not tested (claims file).

In [ ]:
def sync_epochs(r, sg, rd, min_frac):
    bt = BURSTS[(r, sg, rd)][2]
    if min_frac is not None and len(bt):
        bt = bt[H.burst_synchrony(SORT[r]['trains'], bt, SORT[r]['length_ms']) > min_frac]
    ep = np.split(bt, np.where(np.diff(bt) >= 5.0)[0] + 1) if len(bt) else []
    return bt, [e[-1] - e[0] for e in ep]

rows = []
for min_frac in (0.5, 0.4, 0.6, None):
    for sg, rd in READINGS:
        row = dict(units_rising='all bursts' if min_frac is None else f'> {min_frac}', reading=rd, sigma=sg)
        for r in (1, 3, 5, 6, 7, 8):
            bt, spans = sync_epochs(r, sg, rd, min_frac)
            sl = H.RECORDINGS[r]['slice']
            row[sl] = f"{len(bt)}: {sum(sp > 10 for sp in spans)} ({max(spans) if spans else 0:.0f} s)"
            row[sl + '_yes'] = any(sp > 10 for sp in spans)
        row['S3_to_S6_any'] = any(row[k + '_yes'] for k in ('S3', 'S4', 'S5', 'S6'))
        row['S1_and_S2'] = row['S1_yes'] and row['S2_yes']
        rows.append(row)
S4T = pd.DataFrame(rows)
print('cells: bursts kept: epochs longer than 10 s (longest epoch)')
print(S4T[['units_rising', 'reading', 'sigma', 'S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S3_to_S6_any', 'S1_and_S2']].to_string(index=False))
for sg, rd in READINGS:
    print(f'S1 epoch spans (s), units rising > 0.5, ({rd}) sigma {sg}:', np.round(sync_epochs(1, sg, rd, 0.5)[1], 1).tolist())

In [ ]:
# size of the effect: bursts per minute and the threshold relative to the mean rate in each recording
rows = []
for r in (1, 3, 5, 6, 7, 8):
    for sg, rd in READINGS:
        rate, thr, bt, ep = BURSTS[(r, sg, rd)]
        syn = H.burst_synchrony(SORT[r]['trains'], bt, SORT[r]['length_ms']) if len(bt) else np.array([np.nan])
        rows.append(dict(slice=H.RECORDINGS[r]['slice'], reading=rd, sigma=sg, bursts_per_min=len(bt) / (SORT[r]['length_ms'] / 60000),
                         peak_rate_hz=rate.max(), thr_over_mean=thr / rate.mean(), median_frac_units_up=np.nanmedian(syn)))
print(pd.DataFrame(rows).round(2).to_string(index=False))

**Result: partially reproduced, resting on the reading of "sustained".** With a burst counted as synchronized when more than half of the units rise (main), S3 to S6 have no synchronized burst at all under any reading (0 bursts), while S1 has 77 to 138 and S2 89 to 134. S2 has two epochs longer than 10 s (19 s) under every reading; S1 has none, its longest epoch lasting 8 to 10 s. So the test finds no sustained synchronized bursting in S3 to S6, the absence the claim states, but the positive control holds for S2 and not for S1. At a fraction of 0.4, S4 to S6 have 1 to 6 synchronized bursts and no epoch longer than 2 s; at 0.6 none. With every burst kept, as in the first version of this analysis, at least three of S3 to S6 have epochs longer than 10 s (longest 16 to 418 s) and the test does not separate the slices, because those bursts raise the rate of few units (median fraction 0.15 to 0.34, against 0.49 to 0.74 in S1 and S2).

## STTC matrices

STTC with a 20 ms window over the whole sorting (claims file; braingeneers default), for every
recording; used by the next five claims.

In [ ]:
t = time.time()
def sttc_job(r):
    return r, H.sttc(SORT[r]['trains'], SORT[r]['length_ms'], 20.0)
with ProcessPoolExecutor(4, mp_context=mp.get_context('fork')) as ex:
    STTC = dict(ex.map(sttc_job, list(SORT)))
for r, S in STTC.items():
    off = S[~np.eye(len(S), dtype=bool)]
    print(r, H.RECORDINGS[r]['slice'], H.RECORDINGS[r]['cond'], S.shape, f'off-diagonal range {off.min():.3f} to {off.max():.3f}, '
          f'98th percentile {np.percentile(off, 98):.3f}, symmetric {np.allclose(S, S.T)}')
print(f'{time.time()-t:.0f} s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1e6:.1f} GB')

## fig2c-sttc-two-clusters

> Hierarchical clustering of the STTC matrix of S1 gives two groups of units that divide the granule
> cell layer into the same two spatial modes, and S2 gives a similar result.

Code (figures/figure_3/sttc.ipynb): entries of 0.3 or more set to 0 (diagonal included), Euclidean
distances between rows, complete linkage, optimal leaf ordering; cut into 2 clusters (main) and, as
a variant, the two largest clusters of a cut at half the maximum distance; clipping at 0.3 as a
second variant. Unit positions along the layer from the degree-2 fit (`hippo.layer_position`). The
comparison with the phase-cluster split of fig2b, reading (a), uses the code's position thresholds
applied to all units of S1 (group 1: x < 940, or x = 945 with y < 1120, or x > 900 with y < 900).

In [ ]:
def code_group1(pos):
    x, y = pos[:, 0], pos[:, 1]
    return (x < 940) | ((x == 945) & (y < 1120)) | ((x > 900) & (y < 900))

def sttc_clusters(S, mode='zero', cut='maxclust'):
    M_ = S.copy()
    if mode == 'zero':
        M_[M_ >= 0.3] = 0
    else:
        M_ = np.minimum(M_, 0.3)
    Z = hierarchy.linkage(pdist(M_, 'euclidean'), 'complete', optimal_ordering=True)
    if cut == 'maxclust':
        lab = hierarchy.fcluster(Z, 2, 'maxclust')
    else:
        lab = hierarchy.fcluster(Z, 0.5 * Z[:, 2].max(), 'distance')
        big = pd.Series(lab).value_counts().index[:2]
        lab = np.where(np.isin(lab, big), lab, 0)
    return lab

CL = {}
rows = []
for r in (1, 3):
    pos = SORT[r]['pos']; lp = H.layer_position(pos)
    for mode in ('zero', 'clip'):
        for cut in ('maxclust', 'half'):
            lab = sttc_clusters(STTC[r], mode, cut)
            ids = [k for k in pd.Series(lab[lab > 0]).value_counts().index[:2]]
            a, b = (lab == ids[0]), (lab == ids[1]) if len(ids) > 1 else (lab == -1)
            p = stats.mannwhitneyu(lp[a], lp[b]).pvalue if b.sum() else np.nan
            row = dict(slice=H.RECORDINGS[r]['slice'], threshold=mode, cut=cut, n1=int(a.sum()), n2=int(b.sum()),
                       min_frac=min(a.sum(), b.sum()) / len(lab), med_pos1=np.median(lp[a]), med_pos2=np.median(lp[b]) if b.sum() else np.nan,
                       mw_p=p, dropped=int((lab == 0).sum()))
            if r == 1:
                g1 = code_group1(pos)
                # which STTC cluster overlaps code group 1 most, and is it on the same side along the layer?
                ov = [np.mean(g1[a]), np.mean(g1[b])]
                c1 = 0 if ov[0] >= ov[1] else 1
                lower_is_g1 = np.median(lp[g1]) < np.median(lp[~g1])
                lower_is_c1 = [row['med_pos1'], row['med_pos2']][c1] < [row['med_pos1'], row['med_pos2']][1 - c1]
                row.update(code_group1_frac_in_c1=ov[c1], code_group1_frac_in_other=ov[1 - c1], same_side=lower_is_g1 == lower_is_c1)
            row['holds'] = row['min_frac'] >= 0.1 and p < 0.05 and row.get('same_side', True)
            rows.append(row)
            CL[(r, mode, cut)] = lab
F2C = pd.DataFrame(rows)
print(F2C.round(4).to_string(index=False))
print('code group 1 (all units of S1):', int(code_group1(SORT[1]['pos']).sum()), 'of', SORT[1]['n'])

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, r in zip(ax, (1, 3)):
    lab = CL[(r, 'zero', 'maxclust')]; pos = SORT[r]['pos']
    a.scatter(pos[:, 0], pos[:, 1], c=lab, cmap='coolwarm', s=10); a.invert_yaxis(); a.set_aspect('equal')
    a.set_title(f"{H.RECORDINGS[r]['slice']} KA: STTC clusters (2-cluster cut)", fontsize=9)
    print(H.RECORDINGS[r]['slice'], 'x', pos[:, 0].min(), 'to', pos[:, 0].max(), 'y', pos[:, 1].min(), 'to', pos[:, 1].max(), 'um')
plt.tight_layout(); plt.show()

**Result: partially reproduced.** In S2 the two clusters of the 2-cluster cut hold 133 and 47 units (26% the smaller) and differ in position along the layer (median index 77 against 61, Mann-Whitney p = 0.002); the half-distance variant agrees (p = 0.0001) and the clipping variant with the 2-cluster cut fails the 10% size rule (17 units). In S1 the clusters (151 and 61 units) do not differ in position (p = 0.26), and no variant gives p < 0.05 (0.20 to 0.59).

## ext9-sttc-eigenvector-gradient

> The first eigenvector of the STTC matrix, mapped onto the units, forms a gradient across the
> granule cell layer in both S1 and S2, reproducing the hierarchical clusters.

STTC without threshold, first eigenvector by numpy eigh in descending order with the code's sign
convention (human_hip eigenvalues_eigenvectors). Spearman correlation with position along the layer;
Mann-Whitney test of the eigenvector values between the two clusters of fig2c (2-cluster cut,
the code's zeroing).

In [ ]:
def eig_desc(Amat):
    W, U = np.linalg.eigh(Amat)
    U = U * ((-1) ** (U[0, :] < 0))[None, :]
    return W[::-1], U[:, ::-1]
rows = []
for r in (1, 3):
    W, U = eig_desc(STTC[r]); v = U[:, 0]
    lp = H.layer_position(SORT[r]['pos']); lab = CL[(r, 'zero', 'maxclust')]
    rho, p = stats.spearmanr(v, lp)
    ids = pd.Series(lab).value_counts().index[:2]
    pm = stats.mannwhitneyu(v[lab == ids[0]], v[lab == ids[1]]).pvalue
    rows.append(dict(slice=H.RECORDINGS[r]['slice'], eig1=W[0], eig2=W[1], spearman_rho=rho, spearman_p=p, mw_p_clusters=pm,
                     median_v_c1=np.median(v[lab == ids[0]]), median_v_c2=np.median(v[lab == ids[1]]), holds=(p < 0.05) and (pm < 0.05)))
    # variant: second eigenvector (the first is often a common mode)
    rho2, p2 = stats.spearmanr(U[:, 1], lp)
    pm2 = stats.mannwhitneyu(U[lab == ids[0], 1], U[lab == ids[1], 1]).pvalue
    rows[-1].update(v2_spearman_rho=rho2, v2_spearman_p=p2, v2_mw_p=pm2)
    print(H.RECORDINGS[r]['slice'], 'first eigenvector range', v.min().round(3), 'to', v.max().round(3), '; units with v<0:', int((v < 0).sum()))
print(pd.DataFrame(rows).to_string(index=False))

**Result: partially reproduced.** In S2 the first eigenvector correlates with position along the layer (Spearman rho = 0.51, p = 3.6e-13) and differs between the two clusters (p = 7.8e-11). In S1 it differs between the clusters (p = 4.2e-29) but does not correlate with position (rho = 0.08, p = 0.26); there the second eigenvector does (rho = -0.35, p = 2.4e-7), a variant the criterion does not name.

## ext9-sttc-beats-correlation

> For S1, the STTC matrix has larger eigenvalues and a lower reconstruction error than the
> correlation matrix of the spike trains.

Correlation matrix of the 1 ms raster smoothed with a Gaussian of sigma 5 ms (human_hip
correlation_matrix default). Reconstruction error (reading named in the claims file):
||A - A_k||_F / ||A||_F with A_k the rank-k eigen-reconstruction, k = 1 to 20.

In [ ]:
from scipy.ndimage import gaussian_filter1d
s = SORT[1]
nb = int(np.ceil(s['length_ms'])) + 1
raster = np.zeros((s['n'], nb), np.float32)
for i, x in enumerate(s['trains']):
    np.add.at(raster[i], np.clip(x.astype(int), 0, nb - 1), 1)
raster = gaussian_filter1d(raster, 5, axis=1)
CORR = np.corrcoef(raster); del raster
def recon_err(Amat, kmax=20):
    W, U = eig_desc(Amat)
    return W, np.array([np.linalg.norm(Amat - (U[:, :k] * W[:k]) @ U[:, :k].T) / np.linalg.norm(Amat) for k in range(1, kmax + 1)])
Ws, Es = recon_err(STTC[1]); Wc, Ec = recon_err(CORR)
E9 = pd.DataFrame(dict(k=np.arange(1, 21), eig_sttc=Ws[:20], eig_corr=Wc[:20], err_sttc=Es, err_corr=Ec))
print(E9.round(4).to_string(index=False))
print('eigenvalue larger for STTC:', int((E9.eig_sttc > E9.eig_corr).sum()), 'of 20; error lower for STTC:', int((E9.err_sttc < E9.err_corr).sum()), 'of 20')
print('sums of eigenvalues (traces):', round(float(np.trace(STTC[1])), 1), round(float(np.trace(CORR)), 1),
      '; smallest eigenvalues', round(float(Ws.min()), 3), round(float(Wc.min()), 3))

**Result: reproduced.** For S1 each of the first 20 eigenvalues is larger for STTC than for the correlation matrix (16.6 against 8.3 for the first), and the relative reconstruction error is lower for STTC at every k from 1 to 20 (0.70 against 0.87 at k = 1, 0.53 against 0.76 at k = 20). The STTC matrix has negative eigenvalues (smallest -2.64), so it is not positive semidefinite as the Methods state.

## fig3a-significant-pairs

> Of about 45,000 neuron pairs in S1, 784 have a significant directed latency; the other recordings
> give 1940 (S2 KA), 304 (S1 baseline), 37 (S2 baseline), 79 (S3), 37 (S4), 172 (S5) and 92 (S6).

Pair selection as the criterion gives it (`hippo.significant_pairs`).

In [ ]:
t = time.time()
def pairs_job(r):
    return r, H.significant_pairs(SORT[r]['trains'], STTC[r])
with ProcessPoolExecutor(4, mp_context=mp.get_context('fork')) as ex:
    PAIRS = dict(ex.map(pairs_job, list(SORT)))
exp_pairs = {1: 784, 3: 1940, 2: 304, 4: 37, 5: 79, 6: 37, 7: 172, 8: 92}
rows = []
for r in SORT:
    n = SORT[r]['n']; S = STTC[r]
    cand = int((S[np.triu_indices(n, 1)] > 0.01).sum())
    rows.append(dict(rec=r, slice=H.RECORDINGS[r]['slice'], cond=H.RECORDINGS[r]['cond'], units=n, n_squared=n * n,
                     pairs_sttc_above_001=cand, significant=len(PAIRS[r]), expected=exp_pairs[r],
                     within_tol=abs(len(PAIRS[r]) - exp_pairs[r]) <= 0.1 * exp_pairs[r]))
P3 = pd.DataFrame(rows)
print(P3.to_string(index=False))
print('within tolerance:', int(P3.within_tol.sum()), 'of 8;', f'{time.time()-t:.0f} s')

In [ ]:
# sensitivity: the same selection with the STTC threshold and minimum event count changed
for r in (1, 3):
    for smin, mev in [(0.0, 25), (0.01, 10), (0.01, 50)]:
        print(H.RECORDINGS[r]['slice'], H.RECORDINGS[r]['cond'], f'STTC > {smin}, >= {mev} latencies:',
              len(H.significant_pairs(SORT[r]['trains'], STTC[r], sttc_min=smin, min_events=mev)))

**Result: reproduced.** The counts are 784, 304, 1940, 37, 79, 37, 172 and 92 significant pairs, equal to the paper's for all eight recordings. S1 has 212 squared = 44,944 candidate pairs, the "45,000" of the text. Changing the STTC threshold or the minimum number of latencies moves the S1 count to 667 to 841.

## fig3a-bipolar-angle-histograms

> The latency-weighted propagation angle histograms are bipolar for S1 and S2, unimodal for S3 and
> S4, and bimodal but not bipolar for S5 and S6.

Angles from the sending to the receiving unit, weighted by the number of latency events. Measures
named in the claims file: R1 (first-order resultant length), R2 (axial), and the angle between the
two largest peaks of a 16-bin weighted histogram. Peaks are local maxima of the circular histogram.
"One dominant peak" is not defined in the claims file; it is read here as the second-largest local
maximum being below half the largest (a reading of the criterion's words, stated before the result
was seen). Angles are computed with y as stored (image rows) and, as a variant, with y flipped.
Unit positions lie on a 17.5 um grid, so many pair angles fall exactly on the 22.5 degree bin
edges, and flipping y does not simply mirror the histogram: classes can change with the flip. The
second cell counts the angles on bin edges and repeats the classes with the bins shifted by half a
bin (11.25 degrees).

In [ ]:
def angles(r, flip=False):
    P = PAIRS[r]; pos = SORT[r]['pos'].copy()
    if flip:
        pos[:, 1] = -pos[:, 1]
    d = pos[P.recv.values] - pos[P.send.values]
    return np.arctan2(d[:, 1], d[:, 0]), P.n_events.values.astype(float)

def hist_shape(th, w, nbins=16, offset=0.0):
    R1 = np.abs(np.sum(w * np.exp(1j * th))) / w.sum()
    R2 = np.abs(np.sum(w * np.exp(2j * th))) / w.sum()
    h, edges = np.histogram(np.mod(th - offset, 2 * np.pi), bins=nbins, range=(0, 2 * np.pi), weights=w)
    peaks = [i for i in range(nbins) if h[i] > h[i - 1] and h[i] >= h[(i + 1) % nbins]]
    peaks = sorted(peaks, key=lambda i: -h[i])
    centers = (edges[:-1] + edges[1:]) / 2 + offset
    sep = np.nan; ratio = 0.0
    if len(peaks) >= 2:
        dd = np.degrees(abs(centers[peaks[0]] - centers[peaks[1]])) % 360
        sep = min(dd, 360 - dd)
        ratio = h[peaks[1]] / h[peaks[0]]
    if len(peaks) >= 2 and 150 <= sep <= 210 and R2 > R1:
        cls = 'bipolar'
    elif R1 > R2 and (len(peaks) < 2 or ratio < 0.5):
        cls = 'unimodal'
    elif len(peaks) >= 2 and 60 <= sep <= 120:
        cls = 'bimodal not bipolar'
    else:
        cls = 'other'
    return dict(R1=R1, R2=R2, n_peaks=len(peaks), peak_sep_deg=sep, second_over_first=ratio,
                mode1_deg=np.degrees(centers[peaks[0]]) if peaks else np.nan,
                mode2_deg=np.degrees(centers[peaks[1]]) if len(peaks) > 1 else np.nan, cls=cls, hist=h)

want = {1: 'bipolar', 3: 'bipolar', 5: 'unimodal', 6: 'unimodal', 7: 'bimodal not bipolar', 8: 'bimodal not bipolar', 2: 'bipolar', 4: 'bipolar'}
SH = {}
rows = []
for flip in (False, True):
    for r in SORT:
        th, w = angles(r, flip)
        d = hist_shape(th, w); SH[(r, flip)] = d
        rows.append(dict(rec=r, slice=H.RECORDINGS[r]['slice'], cond=H.RECORDINGS[r]['cond'], y_flipped=flip, pairs=len(th),
                         **{k: v for k, v in d.items() if k != 'hist'}, stated=want[r]))
AH = pd.DataFrame(rows)
print(AH.round(3).to_string(index=False))
main = AH[(~AH.y_flipped) & AH.rec.isin([1, 3, 5, 6, 7, 8])]
print('slices in their stated class (S1, S2, S3, S4, S5, S6):', int((main.cls == main.stated).sum()), 'of 6')

In [ ]:
fig, axs = plt.subplots(2, 4, subplot_kw=dict(projection='polar'), figsize=(12, 6))
for a, r in zip(axs.flat, SORT):
    h = SH[(r, False)]['hist']; c = np.linspace(0, 2 * np.pi, 17)[:-1] + np.pi / 16
    a.bar(c, h, width=2 * np.pi / 16); a.set_title(f"{r}: {H.RECORDINGS[r]['slice']} {H.RECORDINGS[r]['cond']}", fontsize=8)
    a.set_yticklabels([])
plt.tight_layout(); plt.show()
print('histogram maxima (latency events per bin):', {r: int(SH[(r, False)]['hist'].max()) for r in SORT})

In [ ]:
# pair angles on the 22.5 degree bin edges, and the class of each slice with the bins as above and shifted by half a
# bin, under both y directions
rows = []
for r in SORT:
    th, w = angles(r)
    deg = np.mod(np.degrees(th), 22.5)
    on = np.isclose(deg, 0, atol=1e-6) | np.isclose(deg, 22.5, atol=1e-6)
    row = dict(rec=r, slice=H.RECORDINGS[r]['slice'], cond=H.RECORDINGS[r]['cond'], pairs=len(th), on_edges=int(on.sum()),
               weight_on_edges=round(float(w[on].sum() / w.sum()), 3))
    for flip in (False, True):
        for off in (0.0, np.pi / 16):
            th, w = angles(r, flip)
            row[f"{'flipped' if flip else 'stored'}_offset_{np.degrees(off):.2f}"] = hist_shape(th, w, offset=off)['cls']
    row['stated'] = want[r]
    rows.append(row)
AH2 = pd.DataFrame(rows)
print(AH2.to_string(index=False))
for col in [c for c in AH2.columns if c.startswith(('stored', 'flipped'))]:
    m = AH2[AH2.rec.isin([1, 3, 5, 6, 7, 8])]
    print(col, '| S1 and S2 bipolar:', bool((m[m.rec.isin([1, 3])][col] == 'bipolar').all()), '| other slices bipolar:',
          int((m[~m.rec.isin([1, 3])][col] == 'bipolar').sum()), '| slices in their stated class:', int((m[col] == m.stated).sum()), 'of 6')

**Result: partially reproduced.** S1 and S2 (KA) are bipolar (modes 180 degrees apart, R2 = 0.52 and 0.68 against R1 = 0.12 and 0.10), and no other slice is. S3 and S4 are not unimodal by the measure used (with y as stored, two peaks 67.5 degrees apart, second peak 0.92 and 0.70 of the first), and S5 and S6 are not bimodal at 60 to 120 degrees (peaks 45 and 135 degrees apart): 2 of 6 slices are in their stated class. 8 to 24% of the latency weight lies on angles exactly on the bin edges. With y flipped or the bins shifted by half a bin, S1 and S2 stay bipolar and no other slice becomes bipolar, while S4, S5 and S6 change class (3 or 4 of 6 in their stated class); S3 and S4 are never unimodal.

## ext10-baseline-same-circuit

> In S1 and S2 the baseline recordings before kainic acid show the same bipolar circuit as the
> recordings during epileptiform activity.

As above on recordings 2 and 4: bipolar, and each mode within 45 degrees of a mode of the KA
recording of the same slice.

In [ ]:
def ang_d(a, b):
    d = abs(a - b) % 360
    return min(d, 360 - d)
for ka, base in [(1, 2), (3, 4)]:
    k, b = SH[(ka, False)], SH[(base, False)]
    md_ = [min(ang_d(m, k['mode1_deg']), ang_d(m, k['mode2_deg'])) for m in (b['mode1_deg'], b['mode2_deg'])]
    print(H.RECORDINGS[ka]['slice'], f"baseline class {b['cls']} (R1 {b['R1']:.3f}, R2 {b['R2']:.3f}, peaks {b['mode1_deg']:.0f} and {b['mode2_deg']:.0f} deg);",
          f"KA class {k['cls']} (peaks {k['mode1_deg']:.0f} and {k['mode2_deg']:.0f} deg); baseline modes from nearest KA mode {md_[0]:.0f} and {md_[1]:.0f} deg;",
          'holds' if b['cls'] == 'bipolar' and max(md_) <= 45 else 'does not hold')

**Result: reproduced.** Both baseline recordings are bipolar (S1 R2 = 0.34 against R1 = 0.08, S2 0.67 against 0.32), and their modes lie 0 and 0 degrees (S1) and 0 and 22 degrees (S2) from the modes of the KA recording of the same slice.

## table3-geometry-test

> The circuit geometry test gives p-values orders of magnitude smaller for the slices with recurrent
> circuits than for the others.

Code (figures/figure_3/haussler/hc112823_acid.ipynb): all significant pairs, sending unit's position
index along the degree-2 fit, angle = arctan2 of receiving minus sending position, Student's t-test
of positions for angle >= 0 against angle < 0. Both directions of the y axis.

In [ ]:
exp_lp = {1: 19.01, 2: 12.07, 3: 67.8, 4: 6.0, 5: 0.92, 6: 0.43, 7: 3.38, 8: 2.96}
rows = []
for flip in (False, True):
    for r in SORT:
        th, w = angles(r, flip)
        lp = H.layer_position(SORT[r]['pos'])[PAIRS[r].send.values]
        g1, g2 = lp[th >= 0], lp[th < 0]
        p = stats.ttest_ind(g1, g2).pvalue if len(g1) > 1 and len(g2) > 1 else np.nan
        rows.append(dict(rec=r, slice=H.RECORDINGS[r]['slice'], cond=H.RECORDINGS[r]['cond'], y_flipped=flip, n_pos=len(g1), n_neg=len(g2),
                         p=p, neglog10p=-np.log10(p), expected=exp_lp[r], within_tol=abs(-np.log10(p) - exp_lp[r]) <= 0.1 * exp_lp[r]))
G3 = pd.DataFrame(rows)
print(G3.to_string(index=False))
for flip in (False, True):
    g = G3[G3.y_flipped == flip].set_index('rec')
    worst_ab = g.loc[[1, 2, 3, 4], 'p'].max(); best_other = g.loc[[5, 6, 7, 8], 'p'].min()
    print(f'y flipped {flip}: within tolerance {int(g.within_tol.sum())} of 8; largest p of S1/S2 {worst_ab:.2e}, smallest p of S3-S6 {best_other:.2e}, ratio {best_other/worst_ab:.1f}')

**Result: reproduced.** With y as stored, -log10 p is 19.02, 12.06, 67.59, 5.98, 0.94, 0.44, 3.38 and 2.97 for recordings 1 to 8, all within tolerance of the paper's (19.01, 12.07, 67.8, 6.0, 0.92, 0.43, 3.38, 2.96). With y flipped 2 of 8 are within tolerance. Under both, the largest p of S1 and S2 is 400 (y as stored) to 4600 (flipped) times smaller than the smallest p of S3 to S6.

## ext12-simulation-recurrence (blocked, not_runnable)

> In a simplified dentate gyrus network model, increasing recurrent connectivity between granule
> cells produces seizure-like bursting.

The criterion asks to run `figures/supplement/simulation/simulation.py` of the paper's repository as
committed. It needs the NEST simulator with a neuron model compiled by NESTML. The simulation's
environment (printed below) pins conda-forge `nest-simulator=3.4.*` and `nestml==5.*`; the registry
runs entries in a pip environment locked with hashes, which cannot hold the conda NEST build. While
preparing the attempt, outside this notebook, the PyPI release of NEST (nest-simulator 3.10.0) was
tried: NESTML 9.0.1 cannot parse the model, whose first line uses the older keyword `neuron`
(printed below; NESTML 9 expects `model`), and NESTML 5 calls `nest.ll_api.sli_func`, which NEST
3.10 no longer has. Those trials need packages outside the entry's lock, so they are not run here
and decide no number; the claim is blocked as `not_runnable`. The sweep (275 simulations of 30 s)
was not timed.

In [ ]:
print(H._fetch('figures/supplement/simulation/environment.yml').decode())
model = H._fetch('figures/supplement/simulation/models/izh_cond_exp2syn.nestml').decode().splitlines()
print('model file, first line:', model[0])

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| sorting-dandiset-link (link) | partially reproduced (6 of 8 recordings found) |
| table1-unit-counts | not reproduced (units 8 of 8; durations on the dandiset 5 of 8) |
| fig1g-ka-synchronized-bursting | not reproduced |
| methods-burst-counts | not reproduced |
| methods-rate-increase-after-ka | not reproduced |
| methods-onset-latency | blocked (metadata_insufficient) |
| s4-ka-no-epileptiform | partially reproduced (resting on an assumption) |
| fig2c-sttc-two-clusters | partially reproduced (S2 holds, S1 does not) |
| ext9-sttc-eigenvector-gradient | partially reproduced (S2 holds, S1 does not) |
| ext9-sttc-beats-correlation | reproduced |
| fig3a-significant-pairs | reproduced (8 of 8 counts equal) |
| fig3a-bipolar-angle-histograms | partially reproduced |
| ext10-baseline-same-circuit | reproduced |
| table3-geometry-test | reproduced (8 of 8) |
| ext12-simulation-recurrence | blocked (not_runnable) |

The other claims are in `lfp_elliott_2026.ipynb` and the attempt record.

How the analysis differs from the paper's: the spike sortings are the paper's own, read from its repository (outside input); bursts, superbursts and their counts use readings named in the claims file, as neither the paper nor its code defines them; the pre-KA reference for burst reading (a) is the separate baseline recording, as no time of KA application is recorded; a synchronized burst is one that raises the rate of most units, and sustained bursting an epoch of such bursts lasting more than 10 s; the measures of cluster position, eigenvector gradient and histogram shape are those the claims file names, not the paper's visual judgement.

What was found about the dataset: the dandiset holds raw voltage only, with no units, no epochs and no record of when kainic acid was applied; its session start times are the conversion date. The paper's analysed recordings 1 to 6 are in it (two of them as parts of longer files), and recordings 7 and 8 (S5 and S6) are not in any file with their electrode layout. Subject identifiers name slices, not patients, and the patient letters differ from the paper's (paper patient A, 52 F, is sub-10F; patient C, 20 M, is sub-1A). Every electrode's location is "unknown". In one second (10 to 11 s, 64 channels) the files sub-3C-1 to sub-3C-4 share 78 to 82% of their samples, which first suggested copies of one recording; over their first 40 s and all 972 channels they are not copies: sub-3C-1 to sub-3C-5 (sub-3C-5 has another length) are saturated at the converter's limits (0 or 1023 counts) at the same seconds (10 to 12 or 13 and 20 to 21 s), where the identical samples lie (26 to 27% there, 4% elsewhere), and outside those seconds their LFP correlates at a median of 0.03 to 0.08 (above 0.5 only in the saturated seconds and up to 4 s after them). Nothing in the files records what caused the saturation.

In [ ]:
print(f'notebook ran in {(time.time()-T0)/60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1e6:.1f} GB')